In [12]:
import gc
import torch
import cv2
from ultralytics import YOLO
import sqlite3
import os
import time
from datetime import datetime

In [13]:
gc.collect()
torch.cuda.empty_cache()
print(torch.cuda.memory_allocated() / 1024**2, "MB")

11.5546875 MB


In [14]:
save_folder = "saved_image"
os.makedirs(save_folder, exist_ok=True)

db_path = "detection_history.db"
conn  = sqlite3.connect(db_path)
cursor = conn.cursor()

In [15]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS knife_alerts (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        timestamp TEXT,
        image_path TEXT,
        confidence REAL
    )
""")
conn.commit()

In [16]:
best_model = YOLO(r"C:\Ml-itsc\MoneyApp\Yolo_knife\runs\detect\train-6\weights\best.pt")

In [17]:
cap = cv2.VideoCapture(0)
last_save_time = 0
coldown = 3.0

while True:

    ret, frame = cap.read()
    results = best_model(frame)

    annotated = results[0].plot()

    for result in results:
        for box in result.boxes:
            conf = float(box.conf)
            print(conf)

            if conf > 0.6:
                print("WARNING: KNIFE DETECTED")
                cv2.putText(
                    img=annotated,
                    text="WARNING: KNIFE DETECTED",
                    org=(50, 80),                    
                    fontFace=cv2.FONT_HERSHEY_SIMPLEX,
                    fontScale=1.0,                   
                    color=(0, 0, 255),               
                    thickness=3,                     
                    lineType=cv2.LINE_AA             
                )

                curr_time = time.time()
                if curr_time - last_save_time >coldown:

                    now = datetime.now()
                    time_str = now.strftime("%Y%m%d_%H%M%S_%f") 
                    image_name = f"knife_{time_str}.jpg"
                    image_path = os.path.join(save_folder, image_name)

                    cv2.imwrite(image_path, annotated)

                    formatted_time = now.strftime("%Y-%m-%d %H:%M:%S")
                    cursor.execute("""
                        INSERT INTO knife_alerts (timestamp, image_path, confidence)
                        VALUES (?, ?, ?)
                    """, (formatted_time, image_path, conf))

                    conn.commit()

                    last_save_time = curr_time



    cv2.imshow("Knife Detector", annotated)

    if cv2.waitKey(1) == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


0: 480x640 (no detections), 17.2ms
Speed: 2.0ms preprocess, 17.2ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 14.4ms
Speed: 1.6ms preprocess, 14.4ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 14.5ms
Speed: 1.4ms preprocess, 14.5ms inference, 1.2ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 11.4ms
Speed: 1.9ms preprocess, 11.4ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 12.6ms
Speed: 1.6ms preprocess, 12.6ms inference, 0.7ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 11.0ms
Speed: 1.5ms preprocess, 11.0ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 11.7ms
Speed: 1.2ms preprocess, 11.7ms inference, 0.9ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 9.6ms
Speed: 1.2ms preprocess, 9.6ms inf